# Gerar Legenda MULTICOR — 5 Idiomas (classificação gramatical por palavra)

Gera o arquivo `.ass` com a legenda colorida — não queima em nenhum vídeo
ainda (isso é o notebook `caption-multicolor-burn.ipynb`, separado de
propósito, pra dar espaço pra correção manual no meio do caminho).

No final, duas ações **separadas**: baixar o `.ass` pra revisar/corrigir,
e (só depois de confirmar que ficou bom) salvar no Drive.


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q stanza kiwipiepy

import shutil, sys
from pathlib import Path

import stanza
from kiwipiepy import Kiwi
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (gravado pelo repositorio-sincronizar)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Rode o repositorio-sincronizar pra criá-lo.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ⚠️ Os módulos abaixo (classificacao.py, classificacao_ko.py, cores.py,
# renderizacao.py) precisam estar na MESMA pasta
# narrated_video/pipeline/modulos/ do Drive, junto com os antigos, pra esse
# copytree acima já trazer eles também. Se der erro de import na célula 4,
# é sinal de que faltou subir algum desses 4 arquivos pro Drive.

kiwi = Kiwi()
print("✅ Stanza e Kiwi prontos")


Mounted at /content/drive
✅ 18 módulos copiados
✅ Stanza e Kiwi prontos


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"

IDIOMA_MESTRE = "en"  # idioma de referência — usa o SRT "whisper" (bruto, não
                      # sincronizado); os outros idiomas usam o conjunto já
                      # sincronizado pelo tempo do mestre (sem sufixo)

IDIOMAS_STANZA = {"pt": "pt", "en": "en", "es": "es", "fr": "fr"}  # idiomas que usam Stanza
IDIOMA_KIWI = "ko"  # idioma que usa Kiwi (só coreano, por enquanto)

BOX_BORDER = 6  # espessura da caixa colorida, em px

print(f"Vídeo: {NOME_ORACAO}")


Vídeo: 40_Matt_02


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. BAIXAR OS SRTs — mestre usa "whisper", os outros usam o          ║
# ║  conjunto JÁ SINCRONIZADO (sem sufixo, gerado pelo                ║
# ║  caption-multilang-generate.ipynb) — não usa mais o "whisper" bruto ║
# ║  dos outros idiomas, que tem timing próprio de cada dublagem e    ║
# ║  fica fora de sincronia por conteúdo.                             ║
# ╚══════════════════════════════════════════════════════════════════╗
from config import PipelineConfig
from drive_utils import DriveClient
from srt_utils import ler_srt

drive_client = DriveClient.get()
legendas_por_idioma_raw = {}

todos_idiomas = list(IDIOMAS_STANZA.keys()) + [IDIOMA_KIWI]
for idioma in todos_idiomas:
    config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE=IDIOMA_MESTRE)

    # mestre = "whisper" (é a própria referência, não passa por sincronização);
    # os outros = sem sufixo (conjunto já sincronizado pelo tempo do mestre)
    if idioma == IDIOMA_MESTRE:
        nome_arquivo = config.nome_srt_whisper(idioma)
    else:
        nome_arquivo = config.nome_srt(idioma)

    destino_local = Path(nome_arquivo)
    ok = drive_client.download(config.pasta_oracao, nome_arquivo, destino_local)
    if not ok:
        print(f"  ⚠️  {idioma.upper()}: não achei '{nome_arquivo}' — pulando")
        continue
    legendas_por_idioma_raw[idioma] = ler_srt(destino_local)
    print(f"  ✅ {idioma.upper()} ({nome_arquivo}): {len(legendas_por_idioma_raw[idioma])} bloco(s)")

if not legendas_por_idioma_raw:
    raise FileNotFoundError("Nenhum SRT encontrado pra nenhum idioma")

if IDIOMA_MESTRE not in legendas_por_idioma_raw:
    print(f"⚠️  O idioma mestre ('{IDIOMA_MESTRE}') não foi encontrado — as legendas dos outros "
          f"idiomas estão sincronizadas em relação a ele, então isso é só um aviso informativo, "
          f"não impede o resto de rodar.")


  ✅ PT: 67 bloco(s)
  ✅ EN: 43 bloco(s)
  ✅ ES: 37 bloco(s)
  ✅ FR: 41 bloco(s)
  ✅ KO: 49 bloco(s)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. CLASSIFICAR CADA BLOCO (Stanza pra PT/EN/ES/FR, Kiwi pro KO) ║
# ║  Se já existir uma classificação salva/corrigida no Drive        ║
# ║  (config.nome_classificacao_multicolor), usa ela em vez de rodar ║
# ║  o Stanza/Kiwi de novo pra esse idioma.                          ║
# ╚══════════════════════════════════════════════════════════════════╝
from classificacao import classificar_palavra_stanza
from classificacao_ko import classificar_pecas_palavra_ko
from renderizacao import (PecaColorida, salvar_classificacao_multicolor,
                          carregar_classificacao_multicolor, classificacao_confere)

def _tentar_carregar_classificacao_salva(idioma):
    """Reaproveita a classificação salva no Drive -- mas só se ela for DESTA
    legenda. O reaproveitamento existe pra preservar correção manual; o que
    ele não pode fazer é ressuscitar o texto de uma versão anterior do SRT.
    As peças carregam o texto, então uma classificação velha faz o vídeo
    exibir as palavras antigas com o SRT novo parado ao lado."""
    nome_arquivo = config.nome_classificacao_multicolor(idioma)
    destino_local = Path(nome_arquivo)
    if not drive_client.download(config.pasta_oracao, nome_arquivo, destino_local):
        return None
    blocos = carregar_classificacao_multicolor(destino_local)
    divergencia = classificacao_confere(blocos, legendas_por_idioma_raw[idioma])
    if divergencia:
        print(f"   ⚠️  {idioma.upper()}: descartando '{nome_arquivo}' — {divergencia}.")
        print(f"      (é de uma versão anterior da legenda; vou classificar de novo)")
        return None
    return blocos

blocos_por_idioma = {}
idiomas_reaproveitados = []
idiomas_a_classificar = []

for idioma in list(IDIOMAS_STANZA.keys()) + [IDIOMA_KIWI]:
    if idioma not in legendas_por_idioma_raw:
        continue
    blocos_salvos = _tentar_carregar_classificacao_salva(idioma)
    if blocos_salvos is not None:
        blocos_por_idioma[idioma] = blocos_salvos
        idiomas_reaproveitados.append(idioma)
        print(f"✅ {idioma.upper()}: classificação já salva reaproveitada "
              f"({config.nome_classificacao_multicolor(idioma)}, {len(blocos_salvos)} bloco(s))")
    else:
        idiomas_a_classificar.append(idioma)

# ── Pipelines Stanza — só pros idiomas que ainda precisam ser classificados ──
pipelines_stanza = {}
for idioma, codigo in IDIOMAS_STANZA.items():
    if idioma not in idiomas_a_classificar:
        continue
    stanza.download(codigo, verbose=False)
    pipelines_stanza[idioma] = stanza.Pipeline(codigo, processors="tokenize,pos,lemma", verbose=False)
    print(f"✅ Pipeline Stanza pronto: {idioma}")

# ── PT / EN / ES / FR — todos iguais, palavra por palavra (a classificação
# ── simplificada não precisa mais de tratamento especial pro francês,
# ── já que substantivo não distingue mais gênero) ──────────────────────
for idioma in ("pt", "en", "es", "fr"):
    if idioma not in idiomas_a_classificar:
        continue
    nlp = pipelines_stanza[idioma]
    blocos = []
    for leg in legendas_por_idioma_raw[idioma]:
        doc = nlp(leg.texto)
        pecas = []
        for sentenca in doc.sentences:
            for palavra in sentenca.words:
                classe = classificar_palavra_stanza(
                    palavra.text, palavra.lemma, palavra.upos, palavra.xpos,
                    palavra.feats or "", idioma,
                )
                pecas.append(PecaColorida(palavra.text, classe))
        blocos.append({"inicio_ms": leg.inicio_ms, "fim_ms": leg.fim_ms, "pecas": pecas})
    blocos_por_idioma[idioma] = blocos

    nome_arquivo = config.nome_classificacao_multicolor(idioma)
    salvar_classificacao_multicolor(blocos, Path(nome_arquivo))
    drive_client.upload(Path(nome_arquivo), config.pasta_oracao, "application/json")
    print(f"✅ {idioma.upper()} classificado: {len(blocos)} bloco(s) (salvo em {nome_arquivo})")

# ── COREANO — peça por peça, com colado_anterior pra não ter espaço dentro
# ── da mesma palavra original ────────────────────────────────────────────
if IDIOMA_KIWI in idiomas_a_classificar:
    blocos = []
    for leg in legendas_por_idioma_raw[IDIOMA_KIWI]:
        resultado = kiwi.analyze(leg.texto)
        tokens = resultado[0][0]
        grupos: dict[tuple, list] = {}
        ordem_grupos = []
        for t in tokens:
            chave = (t.sent_position, t.word_position)
            if chave not in grupos:
                grupos[chave] = []
                ordem_grupos.append(chave)
            grupos[chave].append({"peca": t.form, "classe_kiwi": t.tag})

        pecas = []
        for chave in ordem_grupos:
            pecas_da_palavra = grupos[chave]
            classes = classificar_pecas_palavra_ko(pecas_da_palavra)
            for i, (p, classe) in enumerate(zip(pecas_da_palavra, classes)):
                pecas.append(PecaColorida(p["peca"], classe, colado_anterior=(i > 0)))
        blocos.append({"inicio_ms": leg.inicio_ms, "fim_ms": leg.fim_ms, "pecas": pecas})
    blocos_por_idioma[IDIOMA_KIWI] = blocos

    nome_arquivo = config.nome_classificacao_multicolor(IDIOMA_KIWI)
    salvar_classificacao_multicolor(blocos, Path(nome_arquivo))
    drive_client.upload(Path(nome_arquivo), config.pasta_oracao, "application/json")
    print(f"✅ KO classificado: {len(blocos)} bloco(s) (salvo em {nome_arquivo})")

if idiomas_reaproveitados:
    print(f"\nℹ️  {len(idiomas_reaproveitados)} idioma(s) usaram classificação já salva "
          f"(possivelmente corrigida à mão): {', '.join(i.upper() for i in idiomas_reaproveitados)}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. GERAR O .ASS COM CAIXA COLORIDA                              ║
# ╚══════════════════════════════════════════════════════════════════╝
from renderizacao import gerar_ass

config_render = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE=IDIOMA_MESTRE)

caminho_ass = gerar_ass(blocos_por_idioma, config_render, box_border=BOX_BORDER)
print(f"✅ Legenda gerada: {caminho_ass}")
print("\nRode a célula 6 pra baixar e revisar. Só rode a célula 7 (salvar no Drive) depois de confirmar que ficou bom.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. BAIXAR O .ASS (pra revisar / corrigir manualmente)           ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import files
files.download(str(caminho_ass))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7. SALVAR NO DRIVE — só rode depois de conferir que ficou bom   ║
# ║  (revise o .ass baixado na célula 6 antes de rodar essa aqui)    ║
# ╚══════════════════════════════════════════════════════════════════╝
caminho_no_drive = drive_client.upload(caminho_ass, config_render.pasta_oracao)
print(f"✅ Salvo no Drive: {caminho_no_drive}")
